# Selección de tipo de entidad

Ejecuta un LLM (LLama-3.1-8B-Instruct) con CoT para determinar cual de los múltiples tipos de entidad extraidos de wikidata, describen mejor a una entidad.

In [84]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from tqdm import tqdm

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["HF_TOKEN"] = "key"

assert torch.backends.mps.is_available()
device = torch.device("mps")

In [85]:
OUTPUT_DIR = os.path.join("../Datasets", "entity_types_clean")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [94]:
context = """Tu tarea consiste en seleccionar, para cada "entidad", la mejor etiqueta dentro de las opciones en "superclase" que la describa de forma más simple, clara y general.  

Debes elegir **una sola palabra o frase corta** que capture de manera eficiente la naturaleza de la entidad, evitando redundancias o términos demasiado específicos.  

---

### MODO DE RAZONAMIENTO (ejemplo a seguir)
1. Analiza qué tipo de cosa es la entidad (¿persona, lugar, organización, concepto?).
2. Revisa las superclases y determina cuál describe de forma más directa, simple y general.
3. Si varias opciones son posibles, prefiere:
   - la más general y común;
   - la que se entiende por sí sola sin contexto;
   - la que sería más útil como etiqueta en un grafo de conocimiento.
   - Si la etiqueta correcta es un substring de una opción, puedes elegirla (e.g. Ciudad de Estados Unidos → Ciudad).
4. Evita términos demasiado específicos, técnicos o redundantes. También evita describir algo por su propio nombre.

---

### EJEMPLOS CON RAZONAMIENTO

**Ejemplo 1**
entidad: Reino Unido  
superclase: país, país insular, estado soberano, poder colonial  
razonamiento:  
Reino Unido es un estado compuesto que cumple con las características de un país soberano. “País insular” y “poder colonial” son descripciones históricas o geográficas, pero “país” es la etiqueta más simple y general.  
output: país  

**Ejemplo 2**
entidad: Rafaela  
superclase: municipio, asentamiento, municipio de Argentina, ciudad de Argentina  
razonamiento:  
Rafaela es una localidad urbana dentro de Argentina. “Asentamiento” y “municipio” son más generales, pero “ciudad de Argentina” es la forma más específica y natural que la describe sin redundancia. Sin embargo, “ciudad” es suficiente y más general, lo que permite generar mejores clases. 
output: ciudad

**Ejemplo 3**
entidad: empresario  
superclase: profesión, persona jurídica, ocupación, concepto económico, business and administration professionals  
razonamiento:  
“Empresario” se refiere a una persona que ejerce una actividad económica. No es una persona jurídica, sino una ocupación o rol laboral. “Ocupación” es la etiqueta más general y adecuada.  
output: ocupación  

**Ejemplo 4**
entidad: ingeniero  
superclase: profesión, cargo, trabajador  
razonamiento:  
“Profesión” describe directamente lo que es ser ingeniero, mientras que “cargo” o “trabajador” son categorías más amplias.  
output: profesión  

**Ejemplo 5**
entidad: pintor  
superclase: profesión, Q778000, trabajador de la construcción, menestral  
razonamiento:  
“Pintor” puede ser artístico o técnico, pero en ambos casos es una profesión. “Trabajador de la construcción” es un subconjunto y “menestral” es arcaico.  
output: profesión 
---

### NUEVO CASO

Ahora aplica el mismo razonamiento anterior, pero **sin mostrar el razonamiento**, solo entrega el resultado final.  

entidad: {{entidad}}  
superclase: {{lista_de_superclases}}  

output:
""".strip()

In [87]:
tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    dtype=torch.float16, # Bajé a float16 para que quepa en memoria
    low_cpu_mem_usage=True
).to(device)

if model.generation_config.pad_token_id is None and tok.pad_token_id is None:
    model.generation_config.pad_token_id = tok.eos_token_id

Loading checkpoint shards: 100%|██████████| 4/4 [00:15<00:00,  3.81s/it]


In [92]:
def build_messages(entity: str, superclasses: list[str]) -> list[dict]:
    """Construye el diálogo con system=context y el turno del usuario pidiendo solo la etiqueta."""
    sc_str = ", ".join(superclasses)
    user = (
        f"entidad: {entity}\n"
        f"superclase: {sc_str}\n\n"
        f"output:"
    )
    return [
        {"role": "system", "content": context},
        {"role": "user", "content": user},
    ]

@torch.inference_mode()
def generate_label(entity: str, superclasses: list[str],
                   max_new_tokens: int = 8,
                   temperature: float = 0.2,
                   top_p: float = 0.9) -> str:
    """
    Genera la etiqueta final para una entidad dada su lista de superclases.
    Devuelve solo la etiqueta (sin razonamiento ni texto extra).
    """
    messages = build_messages(entity, superclasses)
    inputs = tok.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,    # añade el marcador de turno de assistant
        return_tensors="pt"
    ).to(device)

    out = model.generate(
        input_ids=inputs,
        attention_mask=(inputs != model.generation_config.pad_token_id),
        max_new_tokens=max_new_tokens,
        do_sample=True if temperature > 0 else False,
        temperature=temperature,
        top_p=top_p,
        eos_token_id=tok.eos_token_id,
        pad_token_id=model.generation_config.pad_token_id,
    )

    # Recorta el prefijo del input para quedarte solo con lo generado
    gen_ids = out[0, inputs.shape[-1]:]
    text = tok.decode(gen_ids, skip_special_tokens=True).strip()
    text = text.replace("output:", "").strip()
    text = text.splitlines()[0].strip()
    return text

In [100]:
data_dir = '../Datasets/entity_types/'
files = [f for f in os.listdir(data_dir) if f.endswith('.tsv')][1:]
print(files)

# Material de trabajo:
dfs_dict = {file_name.split(".")[0]: pd.read_csv(os.path.join(data_dir, file_name), sep="\t") for file_name in files}

['el_salvador.tsv', 'honduras.tsv', 'argentina.tsv', 'colombia.tsv', 'venezuela.tsv', 'guatemala.tsv', 'ecuador.tsv', 'panama.tsv', 'usa.tsv', 'nicaragua.tsv', 'paraguay.tsv', 'costa_rica.tsv', 'chile.tsv', 'mexico.tsv', 'republica_dominicana.tsv', 'peru.tsv']


In [90]:
for country_name, df in dfs_dict.items():
    df["superclases"] = [
        [
            elem.strip()
            for col in [inst, subc]
            if isinstance(col, str)
            for elem in col.split(",")
        ]
        for inst, subc in zip(df["instancia_de"], df["subclase_de"])
    ]
    df.drop(columns=["instancia_de", "subclase_de"], inplace=True)

In [91]:
for country_name, df in tqdm(dfs_dict.items(), desc="Procesando países"):
    print("Pais:", country_name)
    tipo_entidad_list = []
    entidades = df["entidad"].tolist()
    superclases = df["superclases"].tolist()

    for i in range(len(entidades)):
        if len(superclases[i]) > 1:
            tipo_entidad = generate_label(entidades[i], superclases[i])
        else:
            tipo_entidad = superclases[i][0]
        tipo_entidad_list.append(tipo_entidad)

    df["tipo_entidad"] = tipo_entidad_list
    

Procesando países:   0%|          | 0/17 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, p

KeyboardInterrupt: 